In [5]:
import re
from collections import Counter
import pandas as pd
import tools as tools
import numpy as np


In [6]:
import re

# Definir los documentos individualmente
d1 = "We wish efficiency in the implementation for a particular applicarion."
d2 = "the classification methods are on applicaction of Li´s ideas."
d3 = "the classification has not followed any implementation pattern."
d4 = "we have to take care of the impplementation time and implementation efficiency."
d5 = "the efficiency is in terms of importation methods and application methods."

# Función para tokenizar
def tokenize(text):
    text = text.lower()  # Convertir a minúsculas
    text = re.sub(r'[^\w\s]', '', text)  # Eliminar puntuación
    words = text.split()  # Dividir en palabras 
    return [word for word in words if len(word) >= 6]
    
    

# Leer y tokenizar cada documento por separado
documents = {"d1": d1, "d2": d2, "d3": d3,"d4": d4, "d5": d5}
word_dict_per_doc = {}

for key, text in documents.items():
    words = tokenize(text)  # Tokenizar y filtrar palabras
    word_list_per_doc[key] = words  # Guardar la lista de palabras (incluyendo repeticiones)

# Imprimir los diccionarios de palabras con sus respectivas repeticiones por documento
for doc, word_list in word_list_per_doc.items():
    print(f"\nDiccionario de palabras (6 o más letras) para {doc} con repeticiones:")
    for index, word in enumerate(word_list, start=1):
        print(f"{index}: {word}")



NameError: name 'word_list_per_doc' is not defined

# ERROR EL EL D5 SOLO CUENTA UNA VEZ LA PALLABRA METHODS

In [7]:
# Crear un diccionario global con todas las palabras únicas de los tres documentos
global_word_set = set()

# Agregar todas las palabras únicas de cada documento al conjunto global
for text in documents.values():
    global_word_set.update(tokenize(text))

# Ordenar y asignar un índice a cada palabra en el diccionario global
global_word_list = sorted(global_word_set)
global_word_dict = {idx + 1: word for idx, word in enumerate(global_word_list)}

# Imprimir el diccionario global de palabras únicas
print("\nDiccionario global de palabras únicas:")
for index, word in global_word_dict.items():
    print(f"{index}: {word}")



Diccionario global de palabras únicas:
1: applicaction
2: applicarion
3: application
4: classification
5: efficiency
6: followed
7: implementation
8: importation
9: impplementation
10: methods
11: particular
12: pattern


In [10]:
# Crear la matriz A automáticamente a partir de los documentos tokenizados

# Lista global de términos únicos ordenados
global_word_list = sorted(global_word_set)

# Crear una matriz de ceros con dimensiones (número de términos únicos x número de documentos)
A = np.zeros((len(global_word_list), len(documents)), dtype=int)

# Llenar la matriz A con la frecuencia de los términos en cada documento
for j, (doc_key, text) in enumerate(documents.items()):
    word_counts = Counter(tokenize(text))  # Contar palabras en el documento
    for i, word in enumerate(global_word_list):
        A[i, j] = word_counts.get(word, 0)  # Asignar la frecuencia de la palabra

# Mostrar la matriz A generada automáticamente
print("Matriz A (término-documento):")
print(A)



NameError: name 'global_word_set' is not defined

In [18]:
# Definir la consulta
query_text = "gold silver truck"

# Tokenizar la consulta
query_tokens = tokenize(query_text)

# Crear la matriz de consulta q con ceros
q = np.zeros((len(global_word_list), 1), dtype=int)

# Llenar la matriz q con la frecuencia de los términos en la consulta
query_counts = Counter(query_tokens)
for i, word in enumerate(global_word_list):
    q[i, 0] = query_counts.get(word, 0)  # Asignar la frecuencia de la palabra

# Mostrar la matriz de consulta q generada automáticamente
print("Matriz q (consulta):")
print(q)



Matriz q (consulta):
[[0]
 [0]
 [0]
 [0]
 [0]
 [1]
 [0]
 [0]
 [0]
 [1]
 [1]]


# paso 2 decomponer la matriz A en U, S y V
# A = USV^T

In [9]:
# Descomposición en valores singulares (SVD) de la matriz A
U, S, VT = np.linalg.svd(A, full_matrices=False)

# Obtener la matriz V a partir de la transpuesta de V^T
V = VT.T

# Convertir S en una matriz diagonal
S_matrix = np.diag(S)



# Mostrar las matrices resultantes
print("Matriz U:")
print(U)

print("\nMatriz S:")
print(S_matrix)

# Mostrar la matriz V
print("\nMatriz V:")
print(V)

print("\nMatriz V^T:")
print(VT)


NameError: name 'A' is not defined

# Paso 3

In [8]:
# Definir el valor de k para la aproximación de rango 2
k = 2

# Mantener solo las primeras k columnas de U
U_k = U[:, :k]

# Mantener solo las primeras k filas y columnas de S
S_k = np.diag(S[:k])

# Mantener solo las primeras k columnas de V
V_k = V[:, :k]

# Mostrar las matrices resultantes de la aproximación de rango 2
print("Matriz U_k (aproximación de rango 2):")
print(U_k)

print("\nMatriz S_k (aproximación de rango 2):")
print(S_k)

print("\nMatriz V_k (aproximación de rango 2):")
print(V_k)

print("\nMatriz V_k^T (transpuesta de V_k):")
print(V_k.T)


NameError: name 'U' is not defined

# Paso 4

In [24]:
# Paso 4: Obtener las coordenadas de los documentos en el espacio reducido de 2 dimensiones
doc_coordinates = V_k



# Paso 5

In [7]:

# Paso 5: Calcular las nuevas coordenadas de la consulta en el espacio reducido
# Se usa la ecuación q' = q^T U_k S_k^(-1)

# Invertir la matriz S_k
S_k_inv = np.linalg.inv(S_k)

# Transformar el vector de consulta
q_reduced = (q.T @ U_k) @ S_k_inv

# Mostrar los resultados
print("Coordenadas de los documentos en el espacio reducido (Paso 4):")
print(doc_coordinates)

print("\nNueva representación de la consulta en el espacio reducido (Paso 5):")
print(q_reduced)

NameError: name 'S_k' is not defined

# paso 6

In [4]:
# Paso 6: Ordenar los documentos según la similitud del coseno en orden descendente

from sklearn.metrics.pairwise import cosine_similarity

# Calcular la similitud del coseno entre la consulta reducida y los documentos reducidos
similarity_scores = cosine_similarity(q_reduced, doc_coordinates)  # q' vs documentos

# Ordenar los documentos por similitud de coseno en orden descendente
ranked_docs = sorted(doc_similarity.items(), key=lambda x: x[1], reverse=True)

# Mostrar los resultados
print("Ranking de documentos según similitud de coseno:")
for rank, (doc, similarity) in enumerate(ranked_docs, start=1):
    print(f"{rank}. {doc} - Similitud: {similarity:.4f}")

# Mostrar el ranking final en el formato esperado
ranking_str = " > ".join([doc for doc, _ in ranked_docs])
print(f"\nOrden final de documentos: {ranking_str}")


NameError: name 'q_reduced' is not defined